In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip3 install -U transformers

# Basic Python modules
from collections import defaultdict
import random
import pickle

# For downloading large files from Google Drive
# https://github.com/wkentaro/gdown
import gdown

# For working with gzip files
# https://docs.python.org/3/library/gzip.html
import gzip

# For working with JSON files
import json

# For data manipulation and analysis
import pandas as pd
import numpy as np

# For machine learning tools and evaluation
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# For deep learning
# https://pytorch.org/tutorials/beginner/basics/quickstart_tutorial.html
import torch

# For plotting and data visualization
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import ticker
sns.set(style='ticks', font_scale=1.2)


from kaggle_secrets import UserSecretsClient

from huggingface_hub import login
from datasets import load_dataset
 
secrets = UserSecretsClient()
WANDB_API_KEY = secrets.get_secret('WANDB_API_KEY')
HF_TOKEN  	= secrets.get_secret('HF_TOKEN')
 
import os
os.environ['WANDB_API_KEY'] = WANDB_API_KEY
os.environ['HF_TOKEN']      = HF_TOKEN

# Task 2 — Data Preparation & Normalisation


**We load the `dair-ai/emotion` dataset from the Hugging Face Hub, inspect it,clean it, derive the label mapping, and tokenize it for DistilBERT. Thedataset is pulled programmatically with an authenticated token read fromKaggle Secrets — no credentials are hardcoded.**


## 2.1 Load from Hugging Face

**We use the `"split"` configuration, which ships canonical **train / validation
/ test** splits (16k / 2k / 2k). Using the provided splits avoids data leakage
that a manual re-split could introduce. Each row has two fields: `text` (the
message) and `label` (an integer 0–5).**

In [ ]:
hf_token = UserSecretsClient().get_secret("HF_TOKEN")

# 3. Pull the dataset (explicit config + token)
ds = load_dataset("dair-ai/emotion", "split", token=hf_token)

print(ds)
print(ds["train"][0])
print(ds["train"].features["label"].names)

In [ ]:
ds["train"].info

## 2.2 Inspect the raw data


**The dataset has exactly two columns — `text` (input) and `label` (target) —which is all a text-classification task needs. Below we check size, missingvalues, duplicates, class balance, and text length. These observations justify every cleaning and tokenization decision that follows.**


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = ds["train"].to_pandas()
names = ds["train"].features["label"].names

# --- size & structure ---
print("Rows:", len(df), "| Columns:", list(df.columns))
print("\nNulls:\n", df.isnull().sum())
print("\nExact-duplicate texts:", df["text"].duplicated().sum())
print("Empty/whitespace texts:", (df["text"].str.strip() == "").sum())

# --- class distribution (emotion IS imbalanced — this matters) ---
counts = df["label"].value_counts().sort_index()
counts.index = names
print("\nClass distribution:\n", counts)
print("\nClass %:\n", (counts / counts.sum() * 100).round(1))

# --- text length (drives your max_length choice) ---
df["n_words"] = df["text"].str.split().str.len()
print("\nWord-length stats:\n", df["n_words"].describe().round(1))
print("95th percentile words:", df["n_words"].quantile(0.95))

# --- two quick plots for the report ---
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
counts.plot.bar(ax=ax[0], title="Class distribution")
df["n_words"].plot.hist(bins=50, ax=ax[1], title="Text length (words)")
plt.tight_layout(); plt.savefig("eda.png", dpi=120); plt.show()

## 2.3 Data quality findings

- **Nulls:** none in either column.
- **Empty/whitespace texts:** none.
- **Exact duplicates:** 31 of 16,000 (~0.19%) in the train split — negligible.

The data is essentially clean, so cleaning is deliberately minimal: we strip
surrounding whitespace and drop exact-duplicate texts. We do **not** lowercase
manually because the `distilbert-base-uncased` tokenizer already lowercases its
input — doing it twice would be redundant. Inventing further cleaning the data
does not need would be unjustified.

## 2.4 Class distribution — imbalanced

`joy` (33.5%) and `sadness` (29.2%) together make up ~63% of the data, while
`surprise` (3.6%) and `love` (8.2%) are rare — roughly a **9:1** ratio between
the largest and smallest class.

This drives two downstream choices:
1. **Weighted F1** as the primary metric (plain accuracy would flatter a model
   that ignores rare classes).
2. We rely on the dataset's **stratified canonical splits** rather than
   re-splitting, preserving this distribution across train/val/test.

## 2.5 Text length → choosing `max_length`

Word-length stats: mean ≈ 19, 95th percentile = 41, max = 66. These are short
messages. Since sub-word tokenization expands word counts, we set
`max_length=64`, which covers the 95th percentile comfortably while truncating
almost nothing — keeping training fast within Kaggle's GPU limits.

## 2.6 Clean, map labels, and tokenize

We now: (1) clean each split, (2) derive `id2label` from the dataset's
`ClassLabel` feature — read from the data, never hardcoded — and save it as
`id2label.json`, (3) tokenize with `max_length=64`. We rename `label` →
`labels` because the Hugging Face `Trainer` expects that exact column name to
compute loss, and we skip padding here so a data collator can pad each batch
dynamically at train time (faster than padding everything to 64).

## 2.7 Removing cross-split leakage

A check found 11 texts present in both `train` and `test`. Although small
(~0.07% of train), overlapping examples let the model memorise rather than
generalise, inflating evaluation metrics. We remove the overlapping rows from
the **train** split only — the validation and test splits are left intact so
they remain unbiased benchmarks. After removal, train/eval overlap is 0.

In [ ]:
import pandas as pd
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import json

# 1. CLEAN each split (strip whitespace, drop exact duplicates)
def clean_split(dataset, name):
    df = dataset.to_pandas()
    df["text"] = df["text"].str.strip()
    df = df[df["text"] != ""]
    before = len(df)
    df = df.drop_duplicates(subset="text").reset_index(drop=True)
    print(f"  {name}: removed {before - len(df)} duplicate rows")
    return Dataset.from_pandas(df, preserve_index=False)

print("Cleaning splits:")
ds_clean = DatasetDict({s: clean_split(ds[s], s) for s in ds.keys()})

# 2. DE-LEAK: drop train rows whose text appears in val/test (train only)
eval_texts = set(ds_clean["validation"]["text"]) | set(ds_clean["test"]["text"])
before = len(ds_clean["train"])
ds_clean["train"] = ds_clean["train"].filter(lambda r: r["text"] not in eval_texts)
print(f"Removed {before - len(ds_clean['train'])} leaked rows from train")
overlap = set(ds_clean["train"]["text"]) & eval_texts
print("Remaining train/eval overlap:", len(overlap))   # should be 0

# 3. id2label from the dataset (not hardcoded) + save mapping
names = ds["train"].features["label"].names
id2label = {i: n for i, n in enumerate(names)}
label2id = {n: i for i, n in enumerate(names)}
with open("id2label.json", "w") as f:
    json.dump(id2label, f, indent=2)
print("id2label:", id2label)

# 4. TOKENIZE the cleaned + de-leaked data (max_length=64)
MODEL = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=64)
ds_tok = ds_clean.map(tokenize, batched=True)
ds_tok = ds_tok.rename_column("label", "labels")   # Trainer expects "labels"
ds_tok.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# 5. SAVE prepared dataset locally (commit only id2label.json to git)
ds_tok.save_to_disk("emotion_tokenized")
print("Saved tokenized dataset:", ds_tok)



## 2.8 Summary

The dataset is loaded, inspected, lightly cleaned (whitespace + duplicates),
label-mapped, and tokenized. The prepared dataset is saved locally and only
`id2label.json` is committed to the repository. This completes Task 2; the
tokenized dataset and `id2label` mapping feed directly into model training
(Task 4).